In [0]:
processName = dbutils.widgets.get('prm_processName')

# Get last read Date + 1
nextSourceFileDataSQL = f"""SELECT NVL(MAX(PROCESSED_FILE_TABLE_DATE)+1,'2023-01-01') AS NEXT_SOURCE_FILE_DATE 
FROM pricing_analytics.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS
WHERE PROCESS_NAME = '{processName}' AND 
PROCESS_STATUS = 'Completed'"""

In [0]:
#Get date to construct FileName
nextSourceFileDataDF = spark.sql(nextSourceFileDataSQL)
nextSourceFileDataDF.select('NEXT_SOURCE_FILE_DATE').collect()[0]['NEXT_SOURCE_FILE_DATE']


In [0]:
from datetime import datetime

In [0]:

# Define source and destination path variables
dailyPricingSourceBaseURL = 'https://retailpricing.blob.core.windows.net/'
dailyPricingSourceFolder = 'daily-pricing/'
dailyPricingSourceFileDate = datetime.strptime(str(nextSourceFileDataDF.select('NEXT_SOURCE_FILE_DATE').collect()[0]['NEXT_SOURCE_FILE_DATE']), '%Y-%m-%d').strftime('%d%m%Y')
dailyPricingSourceFileName = f"PW_MW_DR_{dailyPricingSourceFileDate}.csv"
print(dailyPricingSourceFileName)


dailyPricingSinkLayerName = 'bronze'
dailyPricingSinkStorageAccountName = 'adlsudadatalakehouseedev'
# You can change here for date partitioning
dailyPricingSinkFolderName = 'daily-pricing'


In [0]:
import pandas as pds
dailyPricingSourceURL = dailyPricingSourceBaseURL + dailyPricingSourceFolder + dailyPricingSourceFileName
print(dailyPricingSourceURL)

In [0]:
#Convert source data csv to Pandas DF

dailyPricingPandasDF = pds.read_csv(dailyPricingSourceURL)
print(dailyPricingPandasDF)

In [0]:
# Convert Pandas DF to Spark DF

dailyPricingSparkDF = spark.createDataFrame(dailyPricingPandasDF)

In [0]:
dailyPricingSinkFolderPath = f'abfss://{dailyPricingSinkLayerName}@{dailyPricingSinkStorageAccountName}.dfs.core.windows.net/{dailyPricingSinkFolderName}'

In [0]:
# Write to Destination folder with a new audit column 

from pyspark.sql.functions import current_timestamp
(
    dailyPricingSparkDF.
    withColumn('source_file_load_date', current_timestamp()).
    write.
    mode('append').
    option('header', 'true').
    csv(dailyPricingSinkFolderPath)
)

In [0]:
processName = 'dailyPricingSourceIngest'
processFileDate = nextSourceFileDataDF.select('NEXT_SOURCE_FILE_DATE').collect()[0]['NEXT_SOURCE_FILE_DATE']
processStatus = 'Completed'


processInsertSQL = f""" INSERT INTO pricing_analytics.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS VALUES ('{processName}', '{processFileDate}', '{processStatus}')"""

spark.sql(processInsertSQL)



In [0]:
%sql
delete FROM pricing_analytics.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS